In [32]:
!pip install streamlit scikit-learn joblib pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 29.4 MB/s eta 0:00:00


In [33]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [34]:
from google.colab import files
uploaded=files.upload()

Saving heart_disease_webapplication.csv to heart_disease_webapplication (2).csv


In [35]:
#1.Load the dataset
df=pd.read_csv("heart_disease_webapplication (2).csv")
df

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,target_binary
0,63.000000,1,1,145.000000,233.000000,1,2,150.000000,0,2.300000,3,0.0,6.0,0,0
1,67.000000,1,4,160.000000,286.000000,0,2,108.000000,1,1.500000,2,3.0,3.0,2,1
2,67.000000,1,4,120.000000,229.000000,0,2,129.000000,1,2.600000,2,2.0,7.0,1,1
3,37.000000,1,3,130.000000,250.000000,0,0,187.000000,0,3.500000,3,0.0,3.0,0,0
4,41.000000,0,2,130.000000,204.000000,0,2,172.000000,0,1.400000,1,0.0,3.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1019,60.072205,1,3,115.422056,181.768595,0,0,128.155858,1,1.001941,2,1.0,7.0,2,1
1020,70.928404,1,2,166.727221,244.993631,1,0,108.481311,0,0.698337,2,1.0,7.0,2,1
1021,57.332875,1,2,105.075177,233.146324,0,2,140.342007,0,1.519138,1,0.0,7.0,2,1
1022,40.881554,0,4,125.068578,154.370678,1,2,123.756294,0,1.936684,2,1.0,7.0,2,1


In [36]:
df.isnull().sum()

,0
age,0
sex,0
cp,0
trestbps,0
chol,0
fbs,0
restecg,0
thalach,0
exang,0
oldpeak,0


In [37]:
#2.Separate input features and target
x=df.drop("target_binary",axis=1)
y=df["target_binary"]
# 3.Split the dataset
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)
#4.create the model
model=make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)



In [38]:
#5.train the model
model.fit(x_train,y_train)
#6.Evaluate the model
predictions=model.predict(x_test)
print("Accuracy:",accuracy_score(y_test,predictions))
print(classification_report(y_test,predictions))

Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       111
           1       1.00      1.00      1.00        94

    accuracy                           1.00       205
   macro avg       1.00      1.00      1.00       205
weighted avg       1.00      1.00      1.00       205



In [39]:
#7.save the trained model
import joblib
joblib.dump(model,"df_model.pkl")
print("model saved successfully")



model saved successfully


In [104]:
%%writefile app.py
import pandas as pd
import streamlit as st
import joblib

st.set_page_config(
    page_title="Heart Disease Prediction",
    page_icon="❤️",
    layout="wide"
)

# load model (cached so it loads only once)
@st.cache_resource
def load_model():
    return joblib.load("df_model.pkl")

model = load_model()

# columns the model was trained on
expected = list(getattr(model, "feature_names_in_", []))

st.title("🧡❤️ Heart Disease Prediction App")
st.write("Enter the required values for an educational "
         "machine learning demonstration.")

st.sidebar.header("Patient Information")

# sidebar input fields
age = st.sidebar.slider("Age", 18, 100, 45)
sex = st.sidebar.selectbox("Sex", [0, 1])
cp = st.sidebar.selectbox("Chest Pain Type", [0, 1, 2, 3])
trestbps = st.sidebar.number_input("Resting Blood Pressure", 80, 250, 120)
chol = st.sidebar.number_input("Cholesterol", 100, 600, 200)
fbs = st.sidebar.selectbox("Fasting Blood Sugar", [0, 1])
restecg = st.sidebar.selectbox("Resting ECG", [0, 1, 2])
thalach = st.sidebar.number_input("Maximum Heart Rate", 60, 250, 150)
exang = st.sidebar.selectbox("Exercise-Induced Angina", [0, 1])
oldpeak = st.sidebar.number_input("ST Depression", min_value=0.0,
                                  max_value=10.0, value=1.0, step=0.1)
slope = st.sidebar.selectbox("ST Slope", [0, 1, 2])
ca = st.sidebar.selectbox("Major Vessels", [0, 1, 2, 3, 4])
thal = st.sidebar.selectbox("Thalassemia Code", [0, 1, 2, 3])
num = st.sidebar.selectbox("num", [0,1,2])

input_data = pd.DataFrame([{
    "age": age,
    "sex": sex,
    "cp": cp,
    "trestbps": trestbps,
    "chol": chol,
    "fbs": fbs,
    "restecg": restecg,
    "thalach": thalach,
    "exang": exang,
    "oldpeak": oldpeak,
    "slope": slope,
    "ca": ca,
    "thal": thal,
    "num": num
}])

st.subheader("Prediction Result")

if st.button("Predict Heart Disease Risk"):

    # check that app columns match the model's training columns
    if expected:
        provided = list(input_data.columns)
        missing = [c for c in expected if c not in provided]
        extra = [c for c in provided if c not in expected]

        if missing or extra:
            st.error(f"Column mismatch. Missing: {missing} | Extra: {extra}")
            st.info(f"Model was trained on: {expected}")
            st.stop()

        # use the same column order as training
        input_data = input_data[expected]

    prediction = int(model.predict(input_data)[0])
    probabilities = model.predict_proba(input_data)[0]
    classes = list(model.classes_)

    # probability of the positive class
    positive_probability = float(probabilities[classes.index(1)])

    st.metric("Model Estimated Positive-class Probability",
              f"{positive_probability:.2%}")

    if prediction == 1:
        st.warning("The model predicts the positive class.")
    else:
        st.success("The model predicts the negative class.")

    st.caption("Educational demonstration only. "
               "This result is not a medical diagnosis.")

    with st.expander("View Input Features"):
        st.dataframe(input_data, width="stretch")

Overwriting app.py


In [105]:
%%writefile requirements.text

streamlit
scikit-learn
joblib

Overwriting requirements.text


In [106]:

!streamlit run app.py &>/content/logs.txt &

In [110]:
from pyngrok import ngrok
ngrok.set_auth_token("3HwtF9L0VfJWsw94JKS6oUK7XbL_7dQD66jonAmFohWUyzbAX")

In [111]:
!streamlit run app.py &>/content/logs.txt &

In [112]:
from pyngrok import ngrok

public_url=ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://cathedral-occupancy-ellipse.ngrok-free.dev" -> "http://localhost:8501"
